## API Call

In [1]:

import re
import unicodedata
import requests
import pandas as pd


def get_nih_awards(full_name, rpp=25, name_aliases=None):

    url = "https://api.reporter.nih.gov/v2/projects/search"

    def clean(value):
        if pd.isna(value):
            return ""
        value = str(value).strip()
        return "" if value.lower() == "nan" else value

    def normalize(value):
        value = unicodedata.normalize("NFKD", clean(value))
        value = "".join(
            c for c in value if not unicodedata.combining(c)
        )
        return re.sub(r"[^a-z0-9]", "", value.casefold())

    full_name = clean(full_name)
    name_parts = full_name.split()

    if len(name_parts) < 2:
        raise ValueError("full_name must contain first and last names.")

    first, last = name_parts[0], name_parts[-1]
    middle = " ".join(name_parts[1:-1])

    if not isinstance(rpp, int) or isinstance(rpp, bool) or rpp < 1:
        raise ValueError("rpp must be a positive integer.")

    limit = min(rpp, 500)

    search_names = [
        {"first_name": first, "last_name": last}
    ]

    if middle:
        search_names.append({
            "first_name": first,
            "middle_name": middle,
            "last_name": last
        })

    search_names.extend(
        {"first_name": clean(alias), "last_name": last}
        for alias in (name_aliases or [])
        if clean(alias)
    )

    unique_names = list({
        tuple(sorted(name.items())): name
        for name in search_names
    }.values())

    projects_by_id = {}

    with requests.Session() as session:

        for search_name in unique_names:
            offset = 0

            while True:

                response = session.post(
                    url,
                    json={
                        "criteria": {
                            "pi_names": [search_name],
                            "include_active_projects": True
                        },
                        "offset": offset,
                        "limit": limit
                    },
                    timeout=30
                )

                response.raise_for_status()
                projects = response.json().get("results") or []

                for project in projects:
                    project_id = (
                        project.get("appl_id")
                        or project.get("project_num")
                        or project.get("project_title")
                    )

                    if project_id is not None:
                        projects_by_id[str(project_id)] = project

                if len(projects) < limit:
                    break

                offset += limit

                if offset > 14999:
                    print("Warning: NIH API offset limit reached.")
                    break

    target_first = normalize(first)
    target_middle = normalize(middle)
    target_last = normalize(last)

    first_aliases = {
        target_first,
        *[normalize(alias) for alias in (name_aliases or []) if normalize(alias)]
    }

    def first_name_matches(candidate):
        candidate = normalize(candidate)

        return bool(candidate) and any(
            candidate == alias
            or (len(candidate) == 1 and alias.startswith(candidate))
            or (len(alias) == 1 and candidate.startswith(alias))
            for alias in first_aliases
        )

    def investigator_name(person):
        return " ".join(
            clean(person.get(key))
            for key in ("first_name", "middle_name", "last_name")
            if clean(person.get(key))
        )

    records = []

    for project in projects_by_id.values():

        investigators = project.get("principal_investigators") or []

        matched = [
            person for person in investigators
            if normalize(person.get("last_name")) == target_last
            and first_name_matches(person.get("first_name"))
        ]

        if len(matched) != 1:
            continue

        person = matched[0]
        contact_status = person.get("is_contact_pi")

        if not isinstance(contact_status, bool):
            continue

        person_middle = normalize(person.get("middle_name"))

        middle_status = (
            "Not supplied"
            if not target_middle or not person_middle
            else "Match"
            if target_middle == person_middle
            else "Initial-compatible"
            if target_middle[0] == person_middle[0]
            else "Different"
        )

        records.append({
            "Matched Name": investigator_name(person),
            "Role": "PI" if contact_status else "Co-PI",
            "Participants": "; ".join(
                filter(None, map(investigator_name, investigators))
            ),
            "Title": project.get("project_title"),
            "Abstract": project.get("abstract_text"),
            "Award Type": project.get("activity_code"),
            "Journal": pd.NA,
            "Amount": project.get("award_amount"),
            "NIH Email": pd.NA,
            "Institute": (
                (project.get("organization") or {}).get("org_name")
                or project.get("organization_name")
                or pd.NA
            ),
            "Start Date": project.get("project_start_date"),
            "End Date": project.get("project_end_date")
        })

    df = pd.DataFrame(records)

    df["Full Name"] = full_name

    output_columns = [
        "Full Name", "Participants", "Searched Email",
        "Matched Name", "Role", "Title", "Abstract",
        "Award Type", "Journal", "Amount", "NIH Email",
        "Institute", "Start Date", "End Date"
    ]

    return df.reindex(columns=output_columns)


In [16]:
def make_full_name(first_name, last_name):

    name_parts = [first_name, last_name]

    name_parts = [
        str(x).strip()
        for x in name_parts
        if pd.notna(x)
        and str(x).strip() != ""
        and str(x).strip().lower() != "nan"
    ]

    return " ".join(name_parts)

In [17]:
df = pd.read_csv("mhti_applicants.csv")[["first_name", "last_name"]]

df["name"] = df.apply(
    lambda row: make_full_name(
        row["first_name"],
        row["last_name"]
    ),
    axis=1
)

print(df)

    first_name   last_name              name
0        Elena      Agapie      Elena Agapie
1         Aima    Ahonkhai     Aima Ahonkhai
2     Nicholas       Allan    Nicholas Allan
3     Nicholas       Allen    Nicholas Allen
4        Nabil   Alshurafa   Nabil Alshurafa
..         ...         ...               ...
535      mehdi  boukhechba  mehdi boukhechba
536      Rocio  de la Vega  Rocio de la Vega
537      sarah        gray        sarah gray
538       shan         lin          shan lin
539      James    suliburk    James suliburk

[540 rows x 3 columns]


In [18]:
import time
import requests
import pandas as pd


def get_nih_awards_for_people(
    people,
    delay=0.2,
    rpp=25,
    name_aliases_by_email=None
):

    all_results = []

    for _, person in people.iterrows():

        full_name = person["name"]

        print(f"  Searching {full_name}")
        
        try:
        
            result = get_nih_awards(
                full_name=full_name
            )

            if not result.empty:
                all_results.append(result)

                print(
                    f"  Retrieved {len(result)} matching project(s)"
                )

                # Report records requiring identity or role review.
                if "Match Status" in result.columns:
                    needs_review = result["Match Status"].isin([
                        "No investigator match",
                        "Multiple investigator matches",
                        "Role unverified"
                    ]).sum()

                    if needs_review:
                        print(
                            f"  Warning: {needs_review} project(s) "
                            "need investigator/role verification"
                        )
            else:
                print("  No matching projects found")

        except requests.exceptions.RequestException as e:
            print(
                f"  API error for {full_name}: {e}"
            )

        except Exception as e:
            print(
                f"  Error for {full_name}: {e}"
            )

        time.sleep(delay)

    if not all_results:
        return pd.DataFrame(columns=[
            "Full Name",
            "Searched Email",
            "Match Name",
            "Role",
            "Participants",
            "Title",
            "Abstract",
            "Award Type",
            "Journal",
            "Amount",
            "NIH Email",
            "Institute",
            "Start Date",
            "End Date"
        ])

    return pd.concat(all_results, ignore_index=True)

In [19]:
results = get_nih_awards_for_people(df)

  Searching Elena Agapie
  No matching projects found
  Searching Aima Ahonkhai
  No matching projects found
  Searching Nicholas Allan
  Retrieved 3 matching project(s)
  Searching Nicholas Allen
  Retrieved 2 matching project(s)
  Searching Nabil Alshurafa
  Retrieved 4 matching project(s)
  Searching Ashwin Ashok
  No matching projects found
  Searching Arjun Athreya
  Retrieved 2 matching project(s)
  Searching Sherif Badawy
  No matching projects found
  Searching Kara Bagot
  No matching projects found
  Searching Jeannie Bailey
  Retrieved 1 matching project(s)
  Searching Ryan Bailey
  No matching projects found
  Searching Yael Bar-Zeev
  Retrieved 1 matching project(s)
  Searching John Batsis
  Retrieved 1 matching project(s)
  Searching Laura Bellows
  Retrieved 1 matching project(s)
  Searching Natalie Benda
  No matching projects found
  Searching Jennifer Blain Christen
  No matching projects found
  Searching Luisa Blanco
  No matching projects found
  Searching Nick Bot

In [20]:
results["Role"].unique()

array(['Co-PI', 'PI'], dtype=object)

In [21]:
results

,Full Name,Participants,Searched Email,Matched Name,Role,Title,Abstract,Award Type,Journal,Amount,NIH Email,Institute,Start Date,End Date
0,Nicholas Allan,Nicholas Paul Allan; Melanie Bozzay,NaN,Nicholas Paul Allan,Co-PI,Optimizing Brief Coping Skills Interventions f...,Abstract\nSuicide is a leading cause of death ...,R01,<NA>,585043,<NA>,OHIO STATE UNIVERSITY,2024-12-01T00:00:00,2029-11-30T00:00:00
1,Nicholas Allan,Nicholas Paul Allan; Stephanie Gorka,NaN,Nicholas Paul Allan,Co-PI,Brief Digitally-Enhanced Intervention for Mana...,"ABSTRACT\nHeavy, problematic alcohol use is hi...",R34,<NA>,393750,<NA>,OHIO STATE UNIVERSITY,2024-08-01T00:00:00,2027-06-30T00:00:00
2,Nicholas Allan,Nicholas Paul Allan; NORMAN BRAD SCHMIDT,NaN,Nicholas Paul Allan,Co-PI,"Randomized Clinical Trial of a Brief, Anxiety ...",Anxiety psychopathology is highly prevalent in...,R01,<NA>,734833,<NA>,FLORIDA STATE UNIVERSITY,2023-01-01T00:00:00,2027-12-31T00:00:00
3,Nicholas Allen,NICHOLAS B ALLEN; Susan E Ramsey,NaN,NICHOLAS B ALLEN,Co-PI,The Vira Platform: A Contextually Triggered Ju...,ABSTRACT\nAlthough buprenorphine is a first-li...,UG3,<NA>,605121,<NA>,RHODE ISLAND HOSPITAL,2025-07-15T00:00:00,2027-06-30T00:00:00
4,Nicholas Allen,NICHOLAS B ALLEN,NaN,NICHOLAS B ALLEN,PI,Development and testing of a digitally assiste...,"ABSTRACT\n Despite efforts to prevent suicide,...",R44,<NA>,794102,<NA>,"KSANA HEALTH, INC.",2022-09-01T00:00:00,2026-12-31T00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478,Yaguang Zheng,Soko Setoguchi Iwata; Yaguang Zheng,NaN,Yaguang Zheng,Co-PI,Rutgers-NYU Center for Asian Health Promotion ...,Overall – Abstract In response to RFA-MD-21-00...,P50,<NA>,2269320,<NA>,RUTGERS BIOMEDICAL AND HEALTH SCIENCES,2021-09-24T00:00:00,2027-06-30T00:00:00
479,Yaguang Zheng,Yaguang Zheng,NaN,Yaguang Zheng,PI,Adaptive Dietary Intervention (ADI) Leveraging...,We propose evaluating an adaptive dietary inte...,R01,<NA>,970016,<NA>,NEW YORK UNIVERSITY,2025-09-19T00:00:00,2028-08-31T00:00:00
480,shan lin,Shan Lin,NaN,Shan Lin,PI,Characterizing the role of MARCH5 in apoptosis...,Project Summary\nAcute myeloid leukemia (AML) ...,R00,<NA>,248977,<NA>,SEATTLE CHILDREN'S HOSPITAL,2024-04-01T00:00:00,2027-03-31T00:00:00
481,shan lin,Shan Lin,NaN,Shan Lin,PI,Exploiting FLVCR1-mediated choline transport i...,PROJECT SUMMARY\nAcute myeloid leukemia (AML) ...,R01,<NA>,597193,<NA>,SEATTLE CHILDREN'S HOSPITAL,2026-09-10T00:00:00,2031-08-31T00:00:00


In [22]:
results.to_excel("NIH_raw3.xlsx", index=False)

## Merge with Demographic Data

In [26]:
import pandas as pd

df_nsf = pd.read_excel("NIH_raw3.xlsx")
df_demo = pd.read_csv("mhti_applicants.csv")

df_demo["Full Name"] = df_demo.apply(
    lambda row: make_full_name(
        row["first_name"],
        row["last_name"]
    ),
    axis=1
)

df_demo = df_demo.drop(columns=['first_name', 'last_name'])

In [27]:
df_nsf2 = df_nsf.copy()
df_demo2 = df_demo.copy()

# --- Merge onto NSF data ---

df_nsf2 = df_nsf2.merge(
    df_demo,
    on=["Full Name"],
    how="left"
)

df_nsf2

,Full Name,Participants,Searched Email,Matched Name,Role,Title,Abstract,Award Type,Journal,Amount,...,cohorts_applied,statuses,n_applications,best_committee_score,any_alternate_withdrawn,scored_application,tier,author_id,works_count,openalex_linked_confidently
0,Nicholas Allan,Nicholas Paul Allan; Melanie Bozzay,NaN,Nicholas Paul Allan,Co-PI,Optimizing Brief Coping Skills Interventions f...,Abstract\nSuicide is a leading cause of death ...,R01,NaN,585043.0,...,2018;2021,Recipient;Declined,2,1.3,0,1,D_ambiguous,A5025675962,157.0,0
1,Nicholas Allan,Nicholas Paul Allan; Stephanie Gorka,NaN,Nicholas Paul Allan,Co-PI,Brief Digitally-Enhanced Intervention for Mana...,"ABSTRACT\nHeavy, problematic alcohol use is hi...",R34,NaN,393750.0,...,2018;2021,Recipient;Declined,2,1.3,0,1,D_ambiguous,A5025675962,157.0,0
2,Nicholas Allan,Nicholas Paul Allan; NORMAN BRAD SCHMIDT,NaN,Nicholas Paul Allan,Co-PI,"Randomized Clinical Trial of a Brief, Anxiety ...",Anxiety psychopathology is highly prevalent in...,R01,NaN,734833.0,...,2018;2021,Recipient;Declined,2,1.3,0,1,D_ambiguous,A5025675962,157.0,0
3,Nicholas Allen,NICHOLAS B ALLEN; Susan E Ramsey,NaN,NICHOLAS B ALLEN,Co-PI,The Vira Platform: A Contextually Triggered Ju...,ABSTRACT\nAlthough buprenorphine is a first-li...,UG3,NaN,605121.0,...,2017,Awarded,1,1.0,0,1,D_ambiguous,A5151652082,1.0,0
4,Nicholas Allen,NICHOLAS B ALLEN,NaN,NICHOLAS B ALLEN,PI,Development and testing of a digitally assiste...,"ABSTRACT\n Despite efforts to prevent suicide,...",R44,NaN,794102.0,...,2017,Awarded,1,1.0,0,1,D_ambiguous,A5151652082,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
510,Yaguang Zheng,Yaguang Zheng,NaN,Yaguang Zheng,PI,Adaptive Dietary Intervention (ADI) Leveraging...,We propose evaluating an adaptive dietary inte...,R01,NaN,970016.0,...,2017;2018,Declined;Declined,2,2.2,0,1,D_ambiguous,A5072873255,136.0,0
511,Yaguang Zheng,Yaguang Zheng,NaN,Yaguang Zheng,PI,Adaptive Dietary Intervention (ADI) Leveraging...,We propose evaluating an adaptive dietary inte...,R01,NaN,970016.0,...,2021,Assigned,1,1.4,0,1,D_ambiguous,A5072873255,136.0,0
512,shan lin,Shan Lin,NaN,Shan Lin,PI,Characterizing the role of MARCH5 in apoptosis...,Project Summary\nAcute myeloid leukemia (AML) ...,R00,NaN,248977.0,...,2022,Declined,1,1.0,0,1,D_ambiguous,A5017265376,103.0,0
513,shan lin,Shan Lin,NaN,Shan Lin,PI,Exploiting FLVCR1-mediated choline transport i...,PROJECT SUMMARY\nAcute myeloid leukemia (AML) ...,R01,NaN,597193.0,...,2022,Declined,1,1.0,0,1,D_ambiguous,A5017265376,103.0,0


In [29]:
df_nsf2.to_excel("NIH_demographic3.xlsx", index=False)